### 종합 실습 과제: Fashion-MNIST로 나만의 패션 아이템 생성하기

이제 우리는 GAN의 원리를 배우고 직접 손글씨 숫자를 생성해 보았습니다. 

이번 종합 과제에서는 한 단계 더 나아가, 손글씨보다 복잡한 `Fashion-MNIST 데이터셋`을 사용하여 여러분만의 패션 아이템을 생성하는 GAN 모델을 구축해 보겠습니다.

Fashion-MNIST는 티셔츠, 바지, 스웨터, 신발 등 10가지 종류의 패션 아이템 이미지로 구성되어 있습니다. 

이 과제를 통해 여러분은 GAN 모델을 다른 데이터셋에 적용하고, 하이퍼파라미터를 조정하며 생성 결과를 개선하는 실전 경험을 쌓게 될 것입니다.

`과제 목표:`

1.  Fashion-MNIST 데이터셋에 맞는 GAN 모델(생성자, 판별자)을 설계하고 구현합니다.

2.  적대적 학습 과정을 통해 모델을 훈련시킵니다.
3.  훈련된 생성자를 사용하여 새로운 패션 아이템 이미지를 생성하고, 그 결과를 시각화하여 분석합니다.

In [ ]:
# fashion_gan_assignment.ipynb

# 1. 필요 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# 2. 환경 설정 및 하이퍼파라미터 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NOISE_DIM = 100
IMAGE_SIZE = 28 * 28
BATCH_SIZE = 128
NUM_EPOCHS = 50
LR = 0.0002

# 3. 데이터 준비: Fashion-MNIST
# [문제 1] Fashion-MNIST 데이터셋을 로드하고, 이미지를 -1과 1 사이로 정규화하는 transform을 정의하세요.
# transforms.Normalize의 인자는 평균(mean)과 표준편차(std)입니다.
# (x - mean) / std 공식을 통해 0~1 범위를 -1~1 범위로 바꾸려면 mean=0.5, std=0.5를 사용합니다.
transform_fashion = transforms.Compose([
    # 여기에 코드를 작성하세요
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

fashion_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_fashion)
fashion_loader = DataLoader(fashion_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. 모델 정의: Generator와 Discriminator
# (힌트: 위에서 배운 MNIST GAN 모델 구조를 거의 그대로 사용해도 좋습니다.
#        필요하다면 모델의 용량을 늘리거나 줄여보세요.)

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        # [문제 2] 아래 '...' 부분에 생성자 모델의 구조를 nn.Sequential로 정의하세요.
        # 예시: nn.Linear, nn.LeakyReLU, nn.Tanh 등
        self.model = nn.Sequential(
            # 여기에 코드를 작성하세요
            nn.Linear(NOISE_DIM, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, IMAGE_SIZE),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        # [문제 3] 아래 '...' 부분에 판별자 모델의 구조를 nn.Sequential로 정의하세요.
        # 예시: nn.Linear, nn.LeakyReLU, nn.Sigmoid 등
        self.model = nn.Sequential(
            # 여기에 코드를 작성하세요
            nn.Linear(IMAGE_SIZE, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# 모델, 손실함수, 옵티마이저 초기화
generator = Generator().to(device)
discriminator = Discriminator().to(device)
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=LR)
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR)


# 5. 훈련 루프
# [문제 4] 아래의 훈련 루프에서 비어있는 부분을 채워 판별자와 생성자의 학습 로직을 완성하세요.
print("Starting Training Loop...")
for epoch in range(NUM_EPOCHS):
    for i, (real_images, _) in enumerate(fashion_loader):

        real_labels = torch.ones(real_images.size(0), 1).to(device)
        fake_labels = torch.zeros(real_images.size(0), 1).to(device)
        real_images = real_images.view(real_images.size(0), -1).to(device)

        # --- 판별자(D) 훈련 ---
        optimizer_D.zero_grad()

        real_output = discriminator(real_images)
        loss_D_real = criterion(real_output, real_labels)

        noise = torch.randn(real_images.size(0), NOISE_DIM).to(device)
        fake_images = generator(noise)

        # 여기에 판별자의 가짜 이미지에 대한 손실을 계산하는 코드를 작성하세요.
        # (힌트: fake_images를 detach() 하는 것을 잊지 마세요)
        fake_output = discriminator(fake_images.detach()) # 이 줄을 채워주세요
        loss_D_fake = criterion(fake_output, fake_labels) # 이 줄을 채워주세요

        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        optimizer_D.step()

        # --- 생성자(G) 훈련 ---
        optimizer_G.zero_grad()

        # 여기에 생성자의 손실을 계산하는 코드를 작성하세요.
        # (힌트: 생성자는 판별자가 가짜 이미지를 진짜(1)로 믿게 만들어야 합니다.)
        fake_output = discriminator(fake_images) # 이 줄을 채워주세요
        loss_G = criterion(fake_output, real_labels) # 이 줄을 채워주세요

        loss_G.backward()
        optimizer_G.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss D: {loss_D.item():.4f}, Loss G: {loss_G.item():.4f}')

        # 6. 중간 결과 시각화
        # 10 에포크마다 생성된 이미지를 확인
        generator.eval() # 평가 모드
        with torch.no_grad():
            fixed_noise = torch.randn(16, NOISE_DIM).to(device)
            generated_images = generator(fixed_noise).cpu().view(-1, 28, 28)

            fig, axes = plt.subplots(4, 4, figsize=(8, 8))
            for i, ax in enumerate(axes.flatten()):
                ax.imshow(generated_images[i], cmap='gray')
                ax.axis('off')
            plt.suptitle(f'Generated Images at Epoch {epoch+1}')
            plt.show()
        generator.train() # 다시 학습 모드


print("Training finished!")

# [보너스 과제]
# 1. 하이퍼파라미터 튜닝: NUM_EPOCHS를 100 이상으로 늘려보거나, LR(학습률)을 바꿔보세요. 생성 결과가 어떻게 변하나요?
# 2. 모델 구조 변경: 생성자와 판별자의 레이어 수나 뉴런 수를 변경해보세요. 더 복잡한 모델이 항상 좋은 결과를 낳을까요?
# 3. 다른 데이터셋 도전: CIFAR-10과 같은 컬러 이미지 데이터셋으로 GAN을 구현해보려면 무엇을 바꿔야 할까요? (힌트: 이미지 채널, 모델 구조)